# Titanic Survival Prediction using Machine Learning Pipeline

## NeuroFive ML Internship – Week 4

**Author:** Muhammad Fahad

### Objective

The objective of this project is to build a professional Machine Learning Pipeline using Scikit-learn. The pipeline performs data preprocessing, feature engineering, model training, and prediction in a single workflow while preventing data leakage and making the code reusable.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

import joblib

## Step 1 – Load the Dataset

In this step, we import the Titanic dataset and explore its structure before preprocessing.

In [3]:
df = pd.read_csv("train.csv")

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
print("Shape:", df.shape)

df.info()

Shape: (891, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


## Step 2 – Data Cleaning

Missing values are handled by filling the Age column with its median value and the Embarked column with its mode. The Cabin column is removed because it contains too many missing values.

In [5]:
# Fill missing Age values with median
df["Age"] = df["Age"].fillna(df["Age"].median())

# Fill missing Embarked values with mode
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# Drop Cabin because it has too many missing values
df = df.drop("Cabin", axis=1)

df.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,0
SibSp,0
Parch,0
Ticket,0
Fare,0


## Step 3 – Feature Engineering

Two new features are created:

- **FamilySize** = SibSp + Parch + 1
- **IsAlone** = Indicates whether a passenger was traveling alone.

These engineered features may improve the model's ability to learn survival patterns.

In [6]:
# Feature 1: Family Size
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

# Feature 2: IsAlone
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

df[["SibSp","Parch","FamilySize","IsAlone"]].head()

,SibSp,Parch,FamilySize,IsAlone
0,1,0,2,0
1,1,0,2,0
2,0,0,1,1
3,1,0,2,0
4,0,0,1,1


## Step 4 – Feature Selection

The most relevant features are selected for model training, while the target variable is **Survived**.

In [7]:
features = [
    "Pclass",
    "Sex",
    "Age",
    "Fare",
    "Embarked",
    "FamilySize",
    "IsAlone"
]

X = df[features]
y = df["Survived"]

print(X.head())
print(y.head())

   Pclass     Sex   Age     Fare Embarked  FamilySize  IsAlone
0       3    male  22.0   7.2500        S           2        0
1       1  female  38.0  71.2833        C           2        0
2       3  female  26.0   7.9250        S           1        1
3       1  female  35.0  53.1000        S           2        0
4       3    male  35.0   8.0500        S           1        1
0    0
1    1
2    1
3    1
4    0
Name: Survived, dtype: int64


## Step 5 – Data Preprocessing using ColumnTransformer

Different preprocessing techniques are applied to different feature types.

- Numerical Features → StandardScaler
- Categorical Features → OneHotEncoder

This ensures each feature is processed appropriately.

In [8]:
# Numerical columns
numerical_features = [
    "Age",
    "Fare",
    "FamilySize"
]

# Categorical columns
categorical_features = [
    "Pclass",
    "Sex",
    "Embarked",
    "IsAlone"
]

In [9]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(drop="first"), categorical_features)
    ]
)

preprocessor

ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['Age', 'Fare', 'FamilySize']),
                                ('cat', OneHotEncoder(drop='first'),
                                 ['Pclass', 'Sex', 'Embarked', 'IsAlone'])])

## Step 6 – Build the Machine Learning Pipeline

A Scikit-learn Pipeline combines preprocessing and Logistic Regression into a single workflow. This prevents data leakage and ensures consistent preprocessing during training and prediction.

In [10]:
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000))
    ]
)

pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Age', 'Fare',
                                                   'FamilySize']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first'),
                                                  ['Pclass', 'Sex', 'Embarked',
                                                   'IsAlone'])])),
                ('model', LogisticRegression(max_iter=1000))])

## Step 7 – Split the Dataset

The dataset is divided into:

- 70% Training Data
- 30% Testing Data

This allows the model to learn from one portion of the data and be evaluated on unseen data.

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (623, 7)
Testing: (268, 7)


## Step 8 – Train the Model

The pipeline is trained using the training dataset. During this step, preprocessing and model fitting occur automatically.

In [12]:
pipeline.fit(X_train, y_train)

print("Pipeline trained successfully!")

Pipeline trained successfully!


## Step 9 – Make Predictions

The trained pipeline predicts survival for the testing dataset.

In [13]:
y_pred = pipeline.predict(X_test)

print(y_pred[:10])

[0 0 0 1 1 1 1 0 1 1]


## Step 10 – Model Evaluation

The model is evaluated using:

- Accuracy
- Precision
- Recall
- F1-Score

These metrics help measure the overall performance of the classifier.

In [14]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.8059701492537313

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.88      0.84       157
           1       0.80      0.70      0.75       111

    accuracy                           0.81       268
   macro avg       0.81      0.79      0.80       268
weighted avg       0.81      0.81      0.80       268



## Step 11 – Save the Pipeline

The complete Machine Learning Pipeline is saved using Joblib. This allows the trained model to be reused later without retraining.

In [15]:
joblib.dump(pipeline, "titanic_pipeline.pkl")

print("Pipeline saved successfully!")

Pipeline saved successfully!


In [16]:
loaded_pipeline = joblib.load("titanic_pipeline.pkl")

print("Pipeline loaded successfully!")

Pipeline loaded successfully!


# Conclusion

A complete Machine Learning Pipeline was successfully developed using Scikit-learn. Feature engineering, preprocessing, model training, evaluation, and model saving were integrated into a single workflow. This approach produces cleaner, reusable, and production-ready machine learning code.